## The BPE algorithm
The BPE training algorithm pairs finds the set of N most common tokens in a text by repeatedly finding the most common bigram of tokens ``A B``, and merging them into a new token ``AB`` (while also keeping ``A`` and ``B`` as tokens). The algorithm starts from an initial set of token classes consisting of all unique bytes in the training text, and keeps merging until the token set contains N token classes. 

Some restrictions are: The algorithm won't merge across spaces (i.e. never create tokens like ``e t``), and will keep punctuation symbols as separate tokens (i.e. never create tokens like ``e.``).

For instance, in the text "then the thief stole the thing", the three first merges would be:
 - ``t,h -> th``
 - ``th,e -> the``
 - ``th,i -> thi``

The algorithm proceeds as follows:
```
Pretokenize the input text (split it on spaces and punctuation signs into words).
Split all words into individual bytes.
Initialize the set of tokens to be all the unique bytes in the input text.

do until the token set contains N tokens:
    Go through all byte-split words to find the most common token bigram A B
    Go through the words once more and replace A B by AB everywhere
    Add AB to the set of tokens
    Add (A B) --> AB to the list of merges
return the set of tokens and the list of merges

```

Below is a skeleton of a Python implementation. The ``train`` method implements the pseudocode above. We are going to develop the remaining methods one by one into a complete implementation in the cells below. 

In [ ]:
import json
import re
from collections import Counter, defaultdict

class TinyStoriesTokenizer:
    def __init__(self, merges={}, vocab=[], ids={}, vocab_size=5000):
        self.vocab_size = vocab_size
        self.merges = merges
        self.vocab = vocab
        self.ids = ids

    # --- PIECEMEAL FUNCTIONS (You will implement these in the cells below) ---

    def _pretokenize(self, text:str) -> list[str]:
        return []

    def _generate_word_frequencies(self, words:list[str]) -> dict[tuple[str], int]:
        return {}

    def _find_most_frequent_token_bigram(self, word_freqs:dict[tuple[str], int]) -> tuple[str, str]:
        return

    def _merge_bigram(self, word_freqs:dict[tuple[str], int], best_pair:tuple[str, str], new_token:str):
        pass

    # --- TRAINING
    def train(self, corpus_path:str):
        with open(corpus_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        words = self._pretokenize(raw_text)
        word_freqs = self._generate_word_frequencies(words)
        
        # Initialize vocabulary with all unique individual characters found
        unique_chars = set()
        for word_tuple in word_freqs:
            for char in word_tuple:
                unique_chars.add(char)
        self.vocab = list(unique_chars)
        
        # Merging loop
        merges = {}  # (bigram) -> priority (lower number -> higher priority)
        num_merges = self.vocab_size - len(self.vocab)
        for i in range(num_merges):
            best_bigram = self._find_most_frequent_token_bigram(word_freqs)   
            self.merges[best_bigram] = i # Rank         
            new_token = "".join(best_bigram)
            self.vocab.append(new_token)
            word_freqs = self._merge_bigram(word_freqs, best_bigram, new_token)
            if (i + 1) % 100 == 0:
                print(f"Merge {i+1}/{num_merges}: {best_bigram} -> {new_token}")
        print(f"Merge {i+1}/{num_merges}: {best_bigram} -> {new_token}")
        self.ids = {v: k for k, v in enumerate(self.vocab)}

    # --- TOKENIZATION (you will implement this some cells below)
    def tokenize(self, text:str) -> (list[str], list[int]):
        return [], []

    # --- SAVING AND LOADING
    def save(self, path):
        serializable_merges = {f"{k[0]}<SPLIT>{k[1]}": v for k, v in self.merges.items()}
        data = {"vocab": self.vocab, "merges": serializable_merges, "vocab_size": self.vocab_size, "ids": self.ids}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        instance = cls(vocab_size=data["vocab_size"])
        instance.vocab = data["vocab"]
        instance.ids = data["ids"]
        instance.merges = {tuple(k.split("<SPLIT>")): v for k, v in data["merges"].items()}
        return instance

### Pretokenization
To improve efficiency and to avoid working with huge list of tokens, we will first pre-tokenize the training text (essentially splitting on spaces and punctuation signs). This is most easily done using a regular expression. 

For example, the sentence "This isn't uncomfortable, is it?" should be pretokenized into

``["This", " isn't", " uncomfortable" "," " is" " it", "?"]``

Note that several tokens contain an initial space. That is a feature and not a bug, since it gives us this nice property: ``text == "".join(pretokenize(text))``.

In [ ]:
def _pretokenize(self, text:str) -> list[str]:
    regexp = r"\w"  # REPLACE WITH YOUR REGULAR EXPRESSION
    tokens = re.findall(regexp, text)
    return tokens

# Attach the _pretokenize function to the tokenizer
TinyStoriesTokenizer._pretokenize = _pretokenize

text = "Once upon a time there was a little girl called Clara. \
Clara was very shy, so she didn't like to talk to anyone. One \
day she was walking in the woods and saw lots of herbs. She \
wanted to try them, so she picked some and took them home. When \
she got home, she said \"What are these herbs?\" Her mom said, \
\"Those herbs can help you feel better!\" Clara was still shy, \
so she didn't want to try them. But her mom said, \"Just try it \
and see how it tastes.\" Clara was still a bit scared, but she \
tried the herbs. She was surprised to find that they were delicious. \
She laughed and said, \"These herbs are really yummy!\" From then on, \
Clara wasn't so shy anymore. She started talking to people and \
trying new things. She thanked the herbs for helping her come out \
of her shell."

tokenizer = TinyStoriesTokenizer()
tokens = tokenizer._pretokenize(text)
print(tokens)

In [ ]:
# Verify -- this cell should evaluate to True
tokens == ['Once', ' upon', ' a', ' time', ' there', ' was', ' a', ' little',
           ' girl', ' called', ' Clara', '.', ' Clara', ' was', ' very',
           ' shy', ',', ' so', ' she', " didn't", ' like', ' to', ' talk',
           ' to', ' anyone', '.', ' One', ' day', ' she', ' was', ' walking',
           ' in', ' the', ' woods', ' and', ' saw', ' lots', ' of', ' herbs',
           '.', ' She', ' wanted', ' to', ' try', ' them', ',', ' so', ' she',
           ' picked', ' some', ' and', ' took', ' them', ' home', '.', ' When',
           ' she', ' got', ' home', ',', ' she', ' said', ' "', 'What', ' are',
           ' these', ' herbs', '?', '"', ' Her', ' mom', ' said', ',', ' "',
           'Those', ' herbs', ' can', ' help', ' you', ' feel', ' better', '!',
           '"', ' Clara', ' was', ' still', ' shy', ',', ' so', ' she',
           " didn't", ' want', ' to', ' try', ' them', '.', ' But', ' her',
           ' mom', ' said', ',', ' "', 'Just', ' try', ' it', ' and', ' see',
           ' how', ' it', ' tastes', '.', '"', ' Clara', ' was', ' still',
           ' a', ' bit', ' scared', ',', ' but', ' she', ' tried', ' the',
           ' herbs', '.', ' She', ' was', ' surprised', ' to', ' find', ' that',
           ' they', ' were', ' delicious', '.', ' She', ' laughed', ' and',
           ' said', ',', ' "', 'These', ' herbs', ' are', ' really', ' yummy',
           '!', '"', ' From', ' then', ' on', ',', ' Clara', " wasn't", ' so',
           ' shy', ' anymore', '.', ' She', ' started', ' talking', ' to',
           ' people', ' and', ' trying', ' new', ' things', '.', ' She',
           ' thanked', ' the', ' herbs', ' for', ' helping', ' her', ' come',
           ' out', ' of', ' her', ' shell', '.']

In [ ]:
# A nice feature of this pretokenization method is that one can recreate the
# input by "".join"tokens)
# Verify -- this cell should evaluate to True
text == "".join(tokens)

### Representing the input text
The next step is to split each pretoken into a tuple of characters, e.g. ``nice -> ("n", "i", "c", "e")``

A lot of words will appear many times in a long text. Therefore, rather than creating a list of all these tuples, we will create a dict ``word_counts`` keeping track of the number of occurrences of each word (or rather, of each tuple of letters).

For instance:

``pretokenize("It's nice, very, nice!") -> ["It's", " nice", ",", " very", " nice", "!"]``

``generate_word_freqs(["It's", " nice", ",", " very", " nice", "!"]) -> 
[(" ", "n", "i", "c", "e"):2, ("I", "t", "'", "s"):1, (","):1, (" ", "v", "e", "r", "y"):1, ("!"):1]``

Your next task is to implement ``generate_word_freqs``.

In [ ]:
def _generate_word_frequencies(self, words:list[str]) -> dict[tuple[str], int]:
    # YOUR CODE HERE
    return None  # REPLACE THIS EXPRESSION WITH YOUR CODE

TinyStoriesTokenizer._generate_word_frequencies = _generate_word_frequencies

# Verify
tokenizer = TinyStoriesTokenizer()
word_freqs = tokenizer._generate_word_frequencies(["It's", " nice", ",", " very", " nice", "!"])
print(word_freqs.get((' ', 'n', 'i', 'c', 'e'), 0) == 2)
print(word_freqs.get(('I', 't', "'", 's'), 0) == 1)
print(word_freqs.get((' ', 'v', 'e', 'r', 'y'), 0) == 1)
print(word_freqs.get((',',), 0) == 1)
print(word_freqs.get(('!',), 0) == 1)

### Finding the most common token bigram

Next, we want to identify the most common token bigram ``(A, B)``, since we later want to merge these two tokens into ``AB`` everywhere that ``A B`` appears.

This identification is done in two steps: First, the function ``count_token_bigrams`` returns a dict, where every key is a pair of tokens (e.g. ``('n', 'i')`` or ``('th', 'e')``), and each value is number of occurrences that token bigram occurs in the input text.

Second, the function ``find_most_frequent_token_bigram`` returns the bigram ``(A, B)``with the highest number of occurrences **and** the new token ``AB`` obtained from this bigram.

In [ ]:
def _count_token_bigrams(self, word_freqs: dict) -> defaultdict(int):
    bigram_counts = defaultdict(int)

    # YOUR CODE HERE
    
    return bigram_counts
    

def _find_most_frequent_token_bigram(self, word_freqs:dict) -> tuple[str, str]:
    bigram_counts = self._count_token_bigrams(word_freqs)
    # If there are several most frequent bigrams, return the first one
    
    best_bigram = (' ', ' ')  # REPLACE WITH YOUR CODE
    
    return best_bigram

TinyStoriesTokenizer._count_token_bigrams = _count_token_bigrams
TinyStoriesTokenizer._find_most_frequent_token_bigram = _find_most_frequent_token_bigram

tokenizer = TinyStoriesTokenizer()

# Verify
word_freqs = tokenizer._generate_word_frequencies(["It's", " nice", ",", " very", " nice-nice", "!"])
token_bigrams = tokenizer._count_token_bigrams(word_freqs)
print(token_bigrams.get(('n', 'i'), 0) == 3)
print(token_bigrams.get(('c', 'e'), 0) == 3)
print(token_bigrams.get(('I', 't'), 0) == 1)
print(token_bigrams.get(("'", 's'), 0) == 1)
best_bigram = tokenizer._find_most_frequent_token_bigram(token_bigrams)
print(best_bigram == ('n', 'i'))

### Merge the most frequent bigram
After having found the most frequently occurring token bigram, the function ``merge_bigram`` goes through every word in the ``word_freqs`` dictionary we constructed previously, and merges all occurrences of the most common bigram ``(A, B)`` into the new token ``AB``.

In [ ]:
def _merge_bigram(self, word_freqs: dict, best_bigram: tuple, new_token: str) -> dict[str, int]:
    new_word_freqs = {}
    for word_tuple, freq in word_freqs.items():
        
        pass  # REPLACE WITH YOUR CODE
        
    return new_word_freqs

TinyStoriesTokenizer._merge_bigram = _merge_bigram

tokenizer = TinyStoriesTokenizer()

# Verify
word_freqs = tokenizer._generate_word_frequencies(["It's", " nice", ",", " very", " nice-nice", "!"])
token_bigrams = tokenizer._count_token_bigrams(word_freqs)
best_bigram = tokenizer._find_most_frequent_token_bigram(token_bigrams)
next_token = "".join(best_bigram)
new_word_freqs = tokenizer._merge_bigram(word_freqs, best_bigram, next_token)
print(new_word_freqs.get((' ', 'ni', 'c', 'e'), 0) == 1)
token_bigrams = tokenizer._count_token_bigrams(new_word_freqs)
print(token_bigrams.get(('ni', 'c'), 0) == 3)
print(('n', 'c') not in token_bigrams)

### Train the tokenizer
Now we are ready to train the tokenizer! Have another look at the 'train' method (at the top of this notebook). It uses your previously implemented methods to do the following:
```
Pretokenize the input text (split it on spaces and punctuation signs into words).
Split all words into individual bytes.
Initialize the set of tokens to be all the unique bytes in the input text.

do until the token set contains N tokens:
    Go through all byte-split words to find the most common token bigram A B
    Go through the words once more and replace A B by AB everywhere
    Add AB to the set of tokens
    Add (A B) --> AB to the list of merges
return the set of tokens and the list of merges
```

In [ ]:
tokenizer = TinyStoriesTokenizer()
tokenizer.train('/datasets/dd2417/tiny_stories_small.txt')

In [ ]:
# Our trained tokenizer now consists of two parts:
# - a mapping from merges to priorities (e.g. ((' t', 'he'), ' the'):37)
# - a mapping from tokens to ids
#
# Let's have a look at a part of these data structures:
merge_list = list(tokenizer.merges.items())
merge_list.sort(key=lambda x:x[1])
print("The 10 merges with the highest priority (lower number=higher priority)")
print("When tokenizing, merges should be applied in order of priority")
for m in merge_list[:10]:
    print(m)
print( "The ids of some random fairy-tale token classes (note the leading space):" )
for t in [' the', ' some', ' all', ' gold', ' silver', ' treasure', ' dragon']:
    print(t, tokenizer.ids[t])
print("Check that the 'ids' and 'tokens' data structures are in sync:")
in_sync = True
for i in range(tokenizer.vocab_size):
    in_sync = in_sync and t == tokenizer.vocab[tokenizer.ids[t]]
print(in_sync)
print("Check that merges have appeared in a logical order:")
print(tokenizer.merges.get((' ', 't'), -1) == 0)
print(tokenizer.ids.get(' t', 0) < tokenizer.ids.get(' the', 0))
print(tokenizer.ids.get(' the', 0) < tokenizer.ids.get(' them', 0))
print(tokenizer.ids.get(' them', 0) < tokenizer.ids.get(" themselves", 0))

### Use the tokenizer


We can now implement the function ``tokenize``, taking a string as input, and computing a list of tokens and a corresponding list of ids.
This tokenization will proceed as follows:
  - pretokenize the string into words
  - split each word into bytes
  - for each word,
      - create all token bigrams (initially token bigrams)
      - find the bigram having the highest priority according to the ``merges`` dict
      - merge the bigram into a new token
      - keep going until no more merges can be applied for the word, then proceed to the next word

In [ ]:
def tokenize(self, text):
    tokens = []
    pretokens = self._pretokenize(text)
    for word in pretokens:
        parts = list(word)
        while len(parts) > 1:

            pass # REPLACE WITH YOUR CODE 
            
        tokens.extend(parts)
    return tokens, [self.ids[t] for t in tokens]

TinyStoriesTokenizer.tokenize = tokenize

# Reuse the merges, vocab, and ids from the tokenizer we just trained
tokenizer = TinyStoriesTokenizer(merges=tokenizer.merges, vocab=tokenizer.vocab, ids=tokenizer.ids)

# Verify
tokens, ids = tokenizer.tokenize("Hi there! That was unfortunate!")
print(tokens == ['Hi', ' there', '!', ' That', ' was', ' un', 'fort', 'un', 'ate', '!'])
print(ids)
print(ids[2] == ids[-1]) if len(ids) > 0 else print(False)

### Save the tokenizer
When everything is implemented and working as it should, run the cell below to save your tokenizer to a file.

In [ ]:
tokenizer.save("tokenizer.json")

### Save the tokenizer class
Run the following cell to save your tokenizer class in a file "tokenizer.py" that you can reuse in the next notebook.

In [ ]:
import inspect
import textwrap

# Define the file header and the base class constructor
header = """import re
import json
from collections import Counter, defaultdict


class TinyStoriesTokenizer:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = []
        self.ids = {} 
               
"""

# List of methods the students implemented
methods_to_export = [
    '_pretokenize', 
    '_generate_word_frequencies', 
    '_count_token_bigrams',
    '_find_most_frequent_token_bigram', 
    '_merge_bigram', 
    'train', 
    'tokenize', 
    'save', 
    'load'
]

with open("tokenizer.py", "w", encoding="utf-8") as f:
    f.write(header)
    for m_name in methods_to_export:
        try:
            # Get the actual function object from the class
            method = getattr(TinyStoriesTokenizer, m_name)
            # Get the source code
            source = textwrap.dedent(inspect.getsource(method))
            # Indent the source code to fit inside the class
            indented_source = textwrap.indent(source, "    ")
            f.write('\n\n' + indented_source)
        except AttributeError:
            print(f"Warning: Method '{m_name}' not found on TinyStoriesTokenizer. Skipping.")

print("tokenizer.py has been generated from your implemented methods.")